# 05 - Rastreabilidade dos gráficos do PPTX
## Tech Challenge Fase 3 — State of Data Brasil (2023, 2024, 2025-2026)

Este notebook existe para **provar a origem de cada número usado nos gráficos**
do material executivo (`presentation/tech_challenge.pptx`).

Estrutura de cada seção abaixo:
1. A consulta (em pandas, equivalente ao SQL rodado no Athena) que gera o número;
2. O resultado real, calculado a partir da camada **Gold** (Parquet);
3. O trecho de código do PPTX (`pptxgenjs`, em JavaScript) que recebeu exatamente
   esse número para desenhar o gráfico.

Isso fecha o ciclo: **Athena/SQL → número real → gráfico no slide**.

In [ ]:
import pandas as pd

TABELAS_GOLD = ["mercado", "remuneracao", "tecnologias", "ia", "diversidade", "trabalho"]
gold = {nome: pd.read_parquet(f"Gold/{nome}", engine="pyarrow") for nome in TABELAS_GOLD}
for nome, df in gold.items():
    print(f"gold_{nome}: {len(df)} linhas")


: 

---
## Gráfico 1 — Slide 8: Distribuição por senioridade (%)

**Consulta equivalente (`sql/mercado.sql`, consulta 2):**
```sql
SELECT ano_pesquisa, senioridade_comparavel, SUM(total_profissionais) AS total,
       ROUND(100.0*SUM(total_profissionais)/SUM(SUM(total_profissionais)) OVER (PARTITION BY ano_pesquisa),1) AS pct
FROM gold_mercado GROUP BY ano_pesquisa, senioridade_comparavel ORDER BY ano_pesquisa
```

In [2]:
df = gold["mercado"]
tab = df.groupby(["ano_pesquisa","senioridade_comparavel"])["total_profissionais"].sum().unstack()
pct = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1)
pct = pct[["Júnior","Pleno","Sênior","Não informado"]]  # ordem usada no grafico
pct


senioridade_comparavel,Júnior,Pleno,Sênior,Não informado
ano_pesquisa,,,,
2023,19.8,26.3,26.8,27.1
2024,16.6,26.4,30.1,26.8
2025-2026,14.8,22.2,34.5,28.4


**Código do gráfico (`build_deck_full.js`, slide 8):**
```javascript
const dataChart = [
  { name:"Junior", labels: anos, values:[19.8,16.6,14.8] },
  { name:"Pleno", labels: anos, values:[26.3,26.4,22.2] },
  { name:"Senior", labels: anos, values:[26.8,30.1,34.5] },
  { name:"Nao informado", labels: anos, values:[27.1,26.8,28.4] },
];
```
✅ Os valores acima batem exatamente com a tabela `pct` calculada na célula anterior.

---
## Gráfico 2 — Slide 9: Top 4 cargos por ano

**Consulta equivalente (`sql/mercado.sql`, consulta 3):**
```sql
SELECT ano_pesquisa, cargo_atual, SUM(total_profissionais) AS total
FROM gold_mercado GROUP BY ano_pesquisa, cargo_atual ORDER BY ano_pesquisa, total DESC
```

In [3]:
df = gold["mercado"]

# Nota: o rotulo exato do cargo mudou de texto entre os anos
# (ex.: 2023 usa "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
# 2024/2025-2026 usam "Engenheiro de Dados/Data Engineer/Data Architect", sem
# "Arquiteto de Dados/"). Por isso a comparacao usa correspondencia por conteudo
# (str.contains) em vez de igualdade exata de string.
principais = ["Analista de Dados", "Cientista de Dados", "Engenheiro de Dados", "Analista de BI"]

linhas = {}
for nome_curto in principais:
    sub = df[df["cargo_atual"].str.contains(nome_curto, na=False)]
    linhas[nome_curto] = sub.groupby("ano_pesquisa")["total_profissionais"].sum()

tabela = pd.DataFrame(linhas).T
tabela

ano_pesquisa,2023,2024,2025-2026
Analista de Dados,907,957,599
Cientista de Dados,687,687,423
Engenheiro de Dados,684,612,402
Analista de BI,506,396,215


**Código do gráfico (slide 9):**
```javascript
const dataChart = [
  { name:"Analista de Dados", labels:anos, values:[907,957,599] },
  { name:"Cientista de Dados", labels:anos, values:[687,687,423] },
  { name:"Engenheiro de Dados", labels:anos, values:[684,612,402] },
  { name:"Analista de BI", labels:anos, values:[506,396,215] },
];
```
✅ Bate com a tabela calculada acima (e também já havia sido confirmado contra print real do Athena).

**Nota de auditoria:** o rótulo de "Engenheiro de Dados" muda de texto entre 2023 e 2024/2025-2026 (a pesquisa alterou a redação da opção) — por isso a consulta usa correspondência por conteúdo (`str.contains`) em vez de igualdade exata, e o resultado segue batendo com o valor usado no gráfico.

---
## Gráfico 3 — Slide 10: Remuneração (faixa mais comum vs. topo da pirâmide)

**Consulta equivalente (`sql/remuneracao.sql`, consulta 5):**
```sql
SELECT faixa_salarial,
    SUM(CASE WHEN ano_pesquisa='2023' THEN total_profissionais ELSE 0 END) AS t2023,
    SUM(CASE WHEN ano_pesquisa='2024' THEN total_profissionais ELSE 0 END) AS t2024,
    SUM(CASE WHEN ano_pesquisa='2025-2026' THEN total_profissionais ELSE 0 END) AS t2025_2026
FROM gold_remuneracao GROUP BY faixa_salarial
```

In [4]:
df = gold["remuneracao"]
evolucao = df.groupby(["faixa_salarial","ano_pesquisa"])["total_profissionais"].sum().unstack()
evolucao.loc[["de R$ 8.001/mês a R$ 12.000/mês", "Acima de R$ 40.001/mês"]]


ano_pesquisa,2023,2024,2025-2026
faixa_salarial,,,
de R$ 8.001/mês a R$ 12.000/mês,1026,1080,707
Acima de R$ 40.001/mês,72,104,115


**Código do gráfico (slide 10):**
```javascript
const dataChart = [
  { name:"R$ 8.001-12.000 (faixa mais comum)", labels:anos, values:[1026,1080,707] },
  { name:"Acima de R$ 40.001 (topo)", labels:anos, values:[72,104,115] },
];
```
✅ Bate com a tabela `evolucao` acima.

---
## Gráfico 4 — Slide 11: Evolução de menções a Python + ranking de clouds

**Consulta equivalente (`sql/tecnologias.sql`, consultas 2 e 5):**
```sql
-- evolução Python
SELECT ano_pesquisa, SUM(total_mencoes) FROM gold_tecnologias
WHERE categoria='linguagem' AND tecnologia='Python' GROUP BY ano_pesquisa;

-- ranking de clouds 2023
SELECT tecnologia, SUM(total_mencoes) FROM gold_tecnologias
WHERE categoria='cloud' AND ano_pesquisa='2023' GROUP BY tecnologia ORDER BY 2 DESC;
```

In [5]:
df = gold["tecnologias"]

python_evolucao = df[(df["categoria"]=="linguagem") & (df["tecnologia"]=="Python")].groupby("ano_pesquisa")["total_mencoes"].sum()
print("Evolução Python:")
print(python_evolucao)
print()

clouds_2023 = df[(df["categoria"]=="cloud") & (df["ano_pesquisa"]=="2023")].groupby("tecnologia")["total_mencoes"].sum().sort_values(ascending=False)
print("Ranking de clouds 2023:")
print(clouds_2023)


Evolução Python:
ano_pesquisa
2023         3299
2024         3040
2025-2026    1928
Name: total_mencoes, dtype: int64

Ranking de clouds 2023:
tecnologia
Amazon Web Services (AWS)                     1529
Azure (Microsoft)                             1141
Google Cloud (GCP)                            1106
Servidores On Premise/Não utilizamos Cloud     582
Cloud Própria                                  251
Oracle Cloud                                   154
IBM                                             47
Snowflake                                        7
Databricks                                       6
Cloudera                                         4
Huawei Cloud                                     3
Digital Ocean                                    2
Denodo                                           1
Azure                                            1
AMT                                              1
Gallery                                          1
NENHUM                        

**Código do gráfico (slide 11):**
```javascript
const dataChart1 = [{ name:"Mencoes a Python", labels:anos, values:[3299,3040,1928] }];
const dataChart2 = [{ name:"Mencoes (2023)", labels:["AWS","Azure","GCP","On Premise","Cloud Propria"],
                       values:[1529,1141,1106,582,251] }];
```
✅ Ambos batem com as células calculadas acima.

---
## Gráfico 5 — Slide 12: Evolução da não-adoção de IA generativa

**Consulta equivalente (`sql/ia.sql`, consulta 1):**
```sql
SELECT ano_pesquisa,
    SUM(CASE WHEN uso_chatgpt_copilot LIKE '%Não utilizo%' THEN total_profissionais ELSE 0 END) AS nao_usa,
    SUM(total_profissionais) AS total
FROM gold_ia GROUP BY ano_pesquisa
```

In [6]:
df = gold["ia"].copy()
df["nao_usa"] = df["uso_chatgpt_copilot"].astype(str).str.contains("Não utilizo")
nao_usa = df[df["nao_usa"]].groupby("ano_pesquisa")["total_profissionais"].sum()
total = df.groupby("ano_pesquisa")["total_profissionais"].sum()
pct_nao_usa = (nao_usa / total * 100).round(1)
pct_nao_usa


ano_pesquisa
2023         19.7
2024          6.5
2025-2026     2.1
Name: total_profissionais, dtype: float64

**Código do gráfico (slide 12):**
```javascript
const dataChart = [{ name:"% que NAO usa nenhuma solucao de IA", labels:anos, values:[19.7,6.5,2.1] }];
```
✅ Bate com `pct_nao_usa` calculado acima.

---
## Gráfico 6 — Slide 13: Participação feminina (%)

**Consulta equivalente (`sql/diversidade.sql`, consulta 1):**
```sql
SELECT ano_pesquisa, genero, SUM(total_profissionais) AS total,
    ROUND(100.0*SUM(total_profissionais)/SUM(SUM(total_profissionais)) OVER (PARTITION BY ano_pesquisa),1) AS pct
FROM gold_diversidade GROUP BY ano_pesquisa, genero
```

In [7]:
df = gold["diversidade"]
tab = df.groupby(["ano_pesquisa","genero"])["total_profissionais"].sum().unstack()
pct = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1)
pct["Feminino"]


ano_pesquisa
2023         24.4
2024         23.5
2025-2026    22.0
Name: Feminino, dtype: float64

**Código do gráfico (slide 13):**
```javascript
const dataChart = [{ name:"% feminino", labels:anos, values:[24.4,23.5,22.0] }];
```
✅ Bate com a coluna `Feminino` calculada acima.

---
## Gráfico 7 — Slide 14: Evolução do modelo de trabalho (%)

**Consulta equivalente (`sql/trabalho.sql`, consulta 1):**
```sql
SELECT ano_pesquisa, modelo_trabalho, SUM(total_profissionais) AS total,
    ROUND(100.0*SUM(total_profissionais)/SUM(SUM(total_profissionais)) OVER (PARTITION BY ano_pesquisa),1) AS pct
FROM gold_trabalho GROUP BY ano_pesquisa, modelo_trabalho
```

In [8]:
df = gold["trabalho"]
tab = df.groupby(["ano_pesquisa","modelo_trabalho"])["total_profissionais"].sum().unstack()
pct = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1)
pct[["Modelo 100% remoto","Modelo 100% presencial","Modelo híbrido com dias fixos de trabalho presencial",
     "Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)"]]


modelo_trabalho,Modelo 100% remoto,Modelo 100% presencial,Modelo híbrido com dias fixos de trabalho presencial,Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)
ano_pesquisa,,,,
2023,41.6,14.9,14.9,18.4
2024,42.6,15.2,16.3,19.1
2025-2026,36.7,19.2,18.5,18.1


**Código do gráfico (slide 14):**
```javascript
const dataChart = [
  { name:"100% remoto", labels:anos, values:[41.6,42.6,36.7] },
  { name:"100% presencial", labels:anos, values:[14.9,15.2,19.2] },
  { name:"Hibrido dias fixos", labels:anos, values:[14.9,16.3,18.5] },
  { name:"Hibrido flexivel", labels:anos, values:[18.4,19.1,18.1] },
];
```
✅ Bate com a tabela `pct` acima.

---
## Gráfico 8 — Slide 15: Diferenças regionais (%)

**Consulta equivalente (`sql/mercado.sql`, consulta 4):**
```sql
SELECT ano_pesquisa, regiao, SUM(total_profissionais) AS total,
    ROUND(100.0*SUM(total_profissionais)/SUM(SUM(total_profissionais)) OVER (PARTITION BY ano_pesquisa),1) AS pct
FROM gold_mercado GROUP BY ano_pesquisa, regiao
```

In [9]:
df = gold["mercado"]
tab = df.groupby(["ano_pesquisa","regiao"])["total_profissionais"].sum().unstack()
pct = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1)
pct[["Sudeste","Sul","Nordeste","Centro-oeste","Norte"]]


regiao,Sudeste,Sul,Nordeste,Centro-oeste,Norte
ano_pesquisa,,,,,
2023,59.9,18.2,11.5,6.5,1.6
2024,60.0,19.8,9.9,6.4,1.2
2025-2026,62.1,15.5,10.9,6.6,1.4


**Código do gráfico (slide 15):**
```javascript
const dataChart = [
  { name:"Sudeste", labels:anos, values:[59.9,60.0,62.1] },
  { name:"Sul", labels:anos, values:[18.2,19.8,15.5] },
  { name:"Nordeste", labels:anos, values:[11.5,9.9,10.9] },
  { name:"Centro-Oeste", labels:anos, values:[6.5,6.4,6.6] },
  { name:"Norte", labels:anos, values:[1.6,1.2,1.4] },
];
```
✅ Bate com a tabela `pct` acima.

---
## Conclusão

Todos os 8 gráficos do material executivo (`tech_challenge.pptx`) têm origem
rastreável: **tabela Gold (Parquet, gerada pelo Glue Job) → consulta SQL/pandas →
número exato usado no código do gráfico**. Nenhum valor foi digitado "de cabeça" —
cada um foi conferido neste notebook contra o dado real.